# 01 — Data Preparation for Mistral Classifier Factory

**Cookbook reference:** [Mistral Classifier Factory — Intent Classification](https://docs.mistral.ai/cookbooks/mistral-classifier_factory-intent_classification)

This notebook prepares **Veridian Systems** historical IT ticket data for fine-tuning
an intent classifier using Mistral's Classifier Factory.
It converts raw tickets to the Classifier Factory JSONL format,
splits the data into train / val / test sets, saves the split files,
and uploads the training and validation sets to the Mistral Files API
ready for the fine-tuning job in `02_train_classifier.ipynb`.

In [ ]:
%pip install -q mistralai pandas scikit-learn python-dotenv rich

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

# ── Paths ─────────────────────────────────────────────────────────────────
# Running in Colab: upload the data/ directory or mount Google Drive and
# set DATA_DIR to the mounted path before continuing.
DATA_DIR   = Path("data")
RAW_FILE   = DATA_DIR / "raw_tickets.json"
TRAIN_FILE = DATA_DIR / "train.jsonl"
VAL_FILE   = DATA_DIR / "val.jsonl"
TEST_FILE  = DATA_DIR / "test.jsonl"

# ── Classifier Factory label key ──────────────────────────────────────────
# The task name used as the key inside the `labels` dict.
# Must be consistent across train, val, test, and inference calls.
LABEL_KEY = "intent"

# ── All valid intent labels (order matters for reproducibility) ───────────
INTENT_LABELS = [
    "access_request",
    "security_incident",
    "hardware_issue",
    "software_issue",
    "onboarding",
    "payments_incident",
    "expense_request",
    "general_question",
]

RANDOM_STATE = 42

## 1. Overview

### What this notebook does

1. Loads `data/raw_tickets.json` — 60 labelled Veridian IT support tickets across
   8 intent categories.
2. Converts each ticket to the **Classifier Factory JSONL format**:
   ```json
   {"text": "<ticket text>", "labels": {"intent": "<category>"}}
   ```
   The nested `labels` object is specific to Classifier Factory and differs from
   standard SFT fine-tuning, which uses a flat `label` field.
3. Splits the data **70 % train / 15 % val / 15 % test**, stratified by intent
   so every class is proportionally represented in all three splits.
4. Saves `data/train.jsonl`, `data/val.jsonl`, and `data/test.jsonl`.
   > **Do not regenerate these files after the classifier has been trained** —
   > changing the split would invalidate test-set evaluation results.
5. Uploads `train.jsonl` and `val.jsonl` to the Mistral Files API and
   prints the file IDs needed for `02_train_classifier.ipynb`.

### Intent categories

| Label | Description |
|---|---|
| `access_request` | Nexus, Prism, GitHub, AWS, Okta provisioning / offboarding |
| `security_incident` | SEV1/SEV2/SEV3 events — phishing, stolen device, suspicious login |
| `hardware_issue` | MacBook M3 MDM, monitors, docks, peripherals, battery |
| `software_issue` | Okta MFA, Slack, Zoom, IDE licences, Docker, Nexus auth |
| `onboarding` | Day-1 setup, Helix on-call, Prism access, badge, VPN |
| `payments_incident` | prod-payments degradation — always P1, paged via Helix |
| `expense_request` | Home office budget, ergonomics, L&D, Adobe, internet stipend |
| `general_question` | Policy queries, Jira routing, WiFi, HR redirects |

## 2. Load and explore raw tickets

In [ ]:
with open(RAW_FILE) as f:
    raw_tickets = json.load(f)

df = pd.DataFrame(raw_tickets)
console.print(f"Loaded [bold]{len(df)}[/bold] tickets across [bold]{df['intent'].nunique()}[/bold] intent categories\n")

# Validate: every ticket must have exactly the fields we need
required_fields = {"id", "text", "intent", "priority"}
missing = [t["id"] for t in raw_tickets if not required_fields.issubset(t.keys())]
if missing:
    raise ValueError(f"Tickets missing required fields: {missing}")

# Validate: all intents are from the known set
unknown_intents = set(df["intent"]) - set(INTENT_LABELS)
if unknown_intents:
    raise ValueError(f"Unknown intent labels in data: {unknown_intents}")

console.print("[green]✓[/green] All fields present and intent labels valid\n")
df[["id", "intent", "priority", "text"]].head(8)

In [ ]:
dist = df["intent"].value_counts().reindex(INTENT_LABELS)

table = Table(title="Class Distribution — raw_tickets.json", show_header=True, header_style="bold")
table.add_column("Intent", style="cyan", min_width=22)
table.add_column("Count", justify="right", style="magenta")
table.add_column("Share", justify="right")
table.add_column("Bar", min_width=20)

for intent, count in dist.items():
    bar = "█" * count
    table.add_row(intent, str(count), f"{count / len(df):.0%}", bar)

console.print(table)
console.print(
    "[dim]Note: 6–8 examples per class is sufficient to demonstrate the pipeline "
    "end-to-end. For production, aim for ≥ 50 examples per class.[/dim]"
)

## 3. Convert to Classifier Factory JSONL format

Following the [Classifier Factory cookbook](https://docs.mistral.ai/cookbooks/mistral-classifier_factory-intent_classification),
each training example must be a JSON object with two fields:

```json
{"text": "<input string>", "labels": {"intent": "<class_name>"}}
```

The `labels` object is a **dict** — the key is the task name (`"intent"`) and the
value is the ground-truth class. This structure supports multi-task classification
(multiple label keys per example), though we use a single task here.

The `text` field is the raw employee request exactly as written — this is what
the agent will pass to the classifier at inference time, so the formats must match.

In [ ]:
def ticket_to_example(ticket: dict) -> dict:
    """Convert a raw ticket dict to a Classifier Factory training example."""
    return {
        "text": ticket["text"],
        "labels": {LABEL_KEY: ticket["intent"]},
    }


examples = [ticket_to_example(t) for t in raw_tickets]

console.print(f"Converted [bold]{len(examples)}[/bold] tickets to Classifier Factory format\n")

console.print("[bold]3 example records:[/bold]")
for ex in examples[:3]:
    console.print(Panel(
        f"[bold cyan]labels:[/bold cyan] {json.dumps(ex['labels'])}\n"
        f"[bold]text:[/bold] {ex['text']}",
        expand=False,
    ))

## 4. Train / validation / test split

Splits: **70 % train · 15 % val · 15 % test**, stratified by intent.

Stratification ensures every intent class appears in all three splits in roughly
the same proportion as in the full dataset — important when some classes
(e.g. `payments_incident`) have fewer examples than others.

The split is done in two passes:
1. Separate 70 % train from 30 % remainder (stratified).
2. Split the 30 % remainder 50/50 into val and test (stratified).

In [ ]:
all_labels = [ex["labels"][LABEL_KEY] for ex in examples]

# Pass 1: 70% train, 30% temp
train_examples, temp_examples = train_test_split(
    examples,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=all_labels,
)

# Pass 2: split temp 50/50 → val (15%) and test (15%)
temp_labels = [ex["labels"][LABEL_KEY] for ex in temp_examples]
val_examples, test_examples = train_test_split(
    temp_examples,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_labels,
)

console.print(
    f"Split complete: "
    f"train=[bold]{len(train_examples)}[/bold]  "
    f"val=[bold]{len(val_examples)}[/bold]  "
    f"test=[bold]{len(test_examples)}[/bold]"
)

In [ ]:
def count_by_label(split: list[dict]) -> dict[str, int]:
    counts: dict[str, int] = {label: 0 for label in INTENT_LABELS}
    for ex in split:
        counts[ex["labels"][LABEL_KEY]] += 1
    return counts


train_counts = count_by_label(train_examples)
val_counts   = count_by_label(val_examples)
test_counts  = count_by_label(test_examples)

table = Table(
    title=f"Examples per split per category  (total {len(examples)})",
    show_header=True,
    header_style="bold",
)
table.add_column("Intent", style="cyan", min_width=22)
table.add_column("Train",  justify="right", style="green")
table.add_column("Val",    justify="right", style="yellow")
table.add_column("Test",   justify="right", style="magenta")
table.add_column("Total",  justify="right")

for label in INTENT_LABELS:
    tr, v, te = train_counts[label], val_counts[label], test_counts[label]
    table.add_row(label, str(tr), str(v), str(te), str(tr + v + te))

table.add_section()
table.add_row(
    "[bold]TOTAL[/bold]",
    f"[bold]{len(train_examples)}[/bold]",
    f"[bold]{len(val_examples)}[/bold]",
    f"[bold]{len(test_examples)}[/bold]",
    f"[bold]{len(examples)}[/bold]",
)

console.print(table)

## 5. Save splits

In [ ]:
def save_jsonl(split: list[dict], path: Path) -> None:
    """Write a list of dicts to a JSONL file, one record per line."""
    with open(path, "w") as f:
        for record in split:
            f.write(json.dumps(record) + "\n")
    console.print(f"  Saved [bold]{len(split):>3}[/bold] examples → [cyan]{path}[/cyan]")


console.print("[bold]Writing JSONL files:[/bold]")
save_jsonl(train_examples, TRAIN_FILE)
save_jsonl(val_examples,   VAL_FILE)
save_jsonl(test_examples,  TEST_FILE)

# ── Spot-check the first line of each file ────────────────────────────────
console.print("\n[bold]Format check (first record of each file):[/bold]")
for path in (TRAIN_FILE, VAL_FILE, TEST_FILE):
    with open(path) as f:
        first = json.loads(f.readline())
    console.print(
        f"  [cyan]{path.name}[/cyan]  "
        f"labels=[green]{first['labels']}[/green]  "
        f"text=[dim]{first['text'][:72]}…[/dim]"
    )

## 6. Forge context

### How this step relates to a production Forge deployment

In a production **Forge** deployment, the domain-adaptation step would not be
a supervised classification fine-tune on labelled tickets.
Instead it would be **continued pre-training on unlabelled enterprise text** —
ticket bodies, internal wiki pages, runbook documents, architecture decision
records, and Slack incident threads — allowing the base model to absorb
Veridian-specific terminology (Nexus, Prism, Helix, prod-payments, SEV tiers)
at the level of the full model weights.

**Classifier Factory is a targeted, supervised variant of the same fine-tuning
infrastructure.** Rather than adapting the entire model to a new text domain,
it fine-tunes the classification head to map input text to a discrete label set.
Both paths share the same underlying infrastructure:

| | Classifier Factory (this demo) | Forge continued pre-training |
|---|---|---|
| **Input data** | Labelled `{text, labels}` pairs | Unlabelled text corpus |
| **Training signal** | Cross-entropy on known labels | Next-token prediction |
| **What adapts** | Classification head (lightweight) | Full model weights |
| **Output** | Intent classifier model ID | Domain-adapted base model |
| **Data residency** | La Plateforme (public) or Forge | Forge (on-prem / private cloud) |
| **Typical data size** | Tens to thousands of examples | Gigabytes of text |

The Python SDK code in this notebook and in `02_train_classifier.ipynb` would
require **only a `server_url` change** to target a Forge endpoint instead of
La Plateforme — the file upload, job creation, and polling logic is identical.

## 7. Upload to Mistral

Upload `train.jsonl` and `val.jsonl` to the Mistral Files API.
The returned file IDs are passed to the fine-tuning job in
`02_train_classifier.ipynb` — copy the config dict printed at the end of this
section into that notebook, or re-run this cell to retrieve fresh IDs.

> `test.jsonl` is **not** uploaded — it is held locally for offline evaluation
> after training, so the model never sees it during fine-tuning.

In [ ]:
from mistralai import Mistral

# ── API key ───────────────────────────────────────────────────────────────
# Colab: add MISTRAL_API_KEY via the Secrets panel (key icon in sidebar).
# Local: add MISTRAL_API_KEY=sk-... to a .env file in this directory.
try:
    from google.colab import userdata  # type: ignore
    MISTRAL_API_KEY = userdata.get("MISTRAL_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")

if not MISTRAL_API_KEY:
    raise EnvironmentError(
        "MISTRAL_API_KEY is not set.\n"
        "Colab: add it via Secrets (key icon in the left sidebar).\n"
        "Local: add MISTRAL_API_KEY=sk-... to a .env file."
    )

# ── Client ────────────────────────────────────────────────────────────────
# To target a Forge endpoint instead of La Plateforme, add:
#   server_url="https://mistral.your-forge-deployment.internal/v1"
client = Mistral(api_key=MISTRAL_API_KEY)
console.print("[bold green]Mistral client initialised.[/bold green]")

In [ ]:
def upload_file(path: Path) -> str:
    """Upload a JSONL file to the Mistral Files API and return its file ID."""
    console.print(f"Uploading [cyan]{path.name}[/cyan] ({path.stat().st_size // 1024} KB) …")
    with open(path, "rb") as f:
        response = client.files.upload(
            file=(path.name, f.read(), "text/plain"),
            purpose="fine-tune",
        )
    console.print(f"  [green]✓[/green] file_id = [bold]{response.id}[/bold]")
    return response.id


train_file_id = upload_file(TRAIN_FILE)
val_file_id   = upload_file(VAL_FILE)

In [ ]:
config = {
    "train_file_id":   train_file_id,
    "val_file_id":     val_file_id,
    "base_model":      "ministral-3b-latest",
    "job_type":        "classifier",
    "label_key":       LABEL_KEY,
    "num_classes":     len(INTENT_LABELS),
    "intent_labels":   INTENT_LABELS,
    "train_examples":  len(train_examples),
    "val_examples":    len(val_examples),
    "test_examples":   len(test_examples),
}

console.print(Panel(
    json.dumps(config, indent=2),
    title="[bold green]Fine-tuning config — copy into 02_train_classifier.ipynb[/bold green]",
    border_style="green",
))

# Also persist to disk so the next notebook can load it without copy-paste
config_path = DATA_DIR / "finetune_config.json"
config_path.write_text(json.dumps(config, indent=2))
console.print(f"\nConfig saved to [cyan]{config_path}[/cyan] for use in Notebook 02.")